BERT를 활용한 문장 분류 실습

Hugging face의 transformers 라이브러리를 사용하여
사전학습된 BERT 모델로 문장 분류를 수행하시오.

다음 문장이 주어져 있을때 각 문장의 감성을 긍정/ 부정으로 분류하시오.

sentences = [
    "이 영화는 정말 재미있었다.",
    "강의 설명은 너무 어려워서 이해하기 힘들었다.",
    "전체적으로 만족스러운 서비스였다." 
    "배송이 너무 늦어서 화가 난다.
]

1. pipeline 사용

In [8]:
from transformers import pipeline

# 1. 모델 불러오기 (pipeline 사용)
classifier = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

# 2. Input 데이터 넣기
inputs = [
    "이 영화는 정말 재미있었다.",
    "강의 설명은 너무 어려워서 이해하기 힘들었다.",
    "전체적으로 만족스러운 서비스였다.",
    "배송이 너무 늦어서 화가 난다."
]

# 3. 결과 확인
results = classifier(inputs)

for i, text in enumerate(inputs):
    result = results[i]
    print(f"Index: {i} | Text: {text}")
    print(f"Result: {result['label']}, Score: {result['score']:.2f}\n")

c:\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mkim\.cache\huggingface\hub\models--tabularisai--multilingual-sentiment-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12473.77it/s]


Index: 0 | Text: 이 영화는 정말 재미있었다.
Result: Positive, Score: 0.91

Index: 1 | Text: 강의 설명은 너무 어려워서 이해하기 힘들었다.
Result: Negative, Score: 0.82

Index: 2 | Text: 전체적으로 만족스러운 서비스였다.
Result: Positive, Score: 0.94

Index: 3 | Text: 배송이 너무 늦어서 화가 난다.
Result: Negative, Score: 0.90



1. AutoTokenizer와 AutoModelForSequenceClassification을 사용할 것
2. 입력 문장을 토큰화 할 것
3. input_ids, attention_mask를 확인할 것
4. 모델 출력 logits를 확인 할 것
5. 최종 예측 클래스를 구할 것 
6. 각 문장 결과를 해석할 것.

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

class Sentiment:
    def __init__(self, model_name = "tabularisai/multilingual-sentiment-analysis"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()
        self.labels = {0: "Very Negative", 1: "Negative", 2: "Neutral", 3: "Positive", 4: "Very Positive"}

    def predict(self, sentence):
        print(f"\n{'*'*50}")
        print(f"입력 문장: {sentence}")
        print(f"{'*'*50}")

        inputs = self.tokenizer(sentence, return_tensors="pt")

        with torch.no_grad():
            outputs = self.model(**inputs)
        
        logits = outputs.logits
        print(f"\n모델 출력 logits:\n {logits.tolist()[0]}")

        probabilities = torch.nn.functional.softmax(logits, dim=-1)
        predicted_idx = torch.argmax(probabilities, dim=-1).item()
        print(f"\n최종 예측 클래스 (Index): {predicted_idx}")

        result = self.labels[predicted_idx]
        return f"이 문장은 {result}"

sentences = [
    "이 영화는 정말 재미있었다.",
    "강의 설명은 너무 어려워서 이해하기 힘들었다.",
    "전체적으로 만족스러운 서비스였다.",
    "배송이 너무 늦어서 화가 난다."
]

sentiment = Sentiment()

for s in sentences:
    res = sentiment.predict(s)
    print(res)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8631.62it/s]



**************************************************
입력 문장: 이 영화는 정말 재미있었다.
**************************************************

모델 출력 logits:
 [-0.9696892499923706, -0.9571951627731323, -1.25575590133667, 2.8662209510803223, -0.36345914006233215]

최종 예측 클래스 (Index): 3
이 문장은 Positive

**************************************************
입력 문장: 강의 설명은 너무 어려워서 이해하기 힘들었다.
**************************************************

모델 출력 logits:
 [0.8006066679954529, 2.688228130340576, -0.967976450920105, -1.131792426109314, -1.332329511642456]

최종 예측 클래스 (Index): 1
이 문장은 Negative

**************************************************
입력 문장: 전체적으로 만족스러운 서비스였다.
**************************************************

모델 출력 logits:
 [-1.1279125213623047, -1.032561182975769, -0.5203285217285156, 3.162992477416992, -1.0453766584396362]

최종 예측 클래스 (Index): 3
이 문장은 Positive

**************************************************
입력 문장: 배송이 너무 늦어서 화가 난다.
**************************************************

모델 출력 logits:
